In [1]:
import os
from pathlib import Path
from typing import List

import torch
from PIL import Image
from torchvision.transforms import functional as F
from torchvision.transforms.functional import InterpolationMode

# ------------------ CONFIG ------------------
IMG_DIR = "/Users/stanimir/Desktop/IBF_masks/data/images"
MASK_DIR = "/Users/stanimir/Desktop/IBF_masks/data/improved_masks"
OUT_IMG_DIR = "/Users/stanimir/Desktop/IBF_masks/data/images_aug"
OUT_MASK_DIR = "/Users/stanimir/Desktop/IBF_masks/data/masks_aug"

ANGLES = [15, 30, 45]

# If True, expand canvas so nothing is cropped (dimensions may change).
EXPAND = True

IMG_EXTS = {".jpg"}   # <--- changed to only .jpg

In [2]:
def load_mask_pt(path: str):
    m = torch.load(path, map_location="cpu")  
    # just in case
    if m.ndim == 3:
        m = m.squeeze()
    m = (m > 0).float()
    return m.unsqueeze(0) 

In [3]:
def rotate_image_and_mask(img: Image.Image, mask_1hw: torch.Tensor, angle: float, expand: bool):
    """
    Rotate PIL image and mask tensor by the same angle.
    - img: PIL.Image (H,W,C)
    - mask_1hw: torch.FloatTensor [1,H,W] in {0,1}
    Returns: (rot_img (PIL), rot_mask_1hw (torch.FloatTensor [1,H',W']))
    """
    rot_img = F.rotate(
        img,
        angle=angle,
        interpolation=InterpolationMode.BILINEAR,
        expand=expand,
        fill=0
    )

    rot_mask = F.rotate(
        mask_1hw,
        angle=angle,
        interpolation=InterpolationMode.NEAREST,
        expand=expand,
        fill=0
    )

    rot_mask = (rot_mask > 0.5).float()
    return rot_img, rot_mask

In [4]:
def main():

    img_dir = Path(IMG_DIR)
    mask_dir = Path(MASK_DIR)
    out_img_dir = Path(OUT_IMG_DIR)
    out_mask_dir = Path(OUT_MASK_DIR)

    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_mask_dir.mkdir(parents=True, exist_ok=True)

    img_files = [p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS]
    img_files.sort()

    if not img_files:
        print(f"No .jpg images found in {IMG_DIR}.")
        return

    print(f"Found {len(img_files)} JPG images. Augmenting with rotations: {ANGLES}")

    for img_path in img_files:
        stem = img_path.stem
        mask_path = mask_dir / f"{stem}_improved.pt"
        if not mask_path.exists():
            print(f"[WARN] Missing mask for image {img_path.name} -> expected {mask_path.name}. Skipping.")
            continue

        img = Image.open(img_path).convert("RGB")
        mask = load_mask_pt(str(mask_path)) 

        for angle in ANGLES:
            rot_img, rot_mask = rotate_image_and_mask(img, mask, angle, EXPAND)

            angle_tag = f"rot{int(angle):03d}"
            out_img_name = f"{stem}_{angle_tag}.jpg"  
            out_mask_name = f"{stem}_{angle_tag}.pt"

            if rot_img.mode != "RGB":
                rot_img = rot_img.convert("RGB")
            rot_img.save(out_img_dir / out_img_name, quality=95)
            torch.save(rot_mask, out_mask_dir / out_mask_name)

        print(f"Augmented {img_path.name} -> {len(ANGLES)} rotations")

    print("Done.")

if __name__ == "__main__":
    main()


Found 496 JPG images. Augmenting with rotations: [15, 30, 45]
Augmented GA 1_PB100636.JPG -> 3 rotations
Augmented GA 1_PB100637.JPG -> 3 rotations


KeyboardInterrupt: 